# Ev Fiyat Tahmini ile Regresyon Analizi (Regression Analysis)

## 1. Konu Anlatımı: Regresyon Analizi Nedir?

**Regresyon Analizi**, bir bağımlı değişken (hedef/target) ile bir veya birden fazla bağımsız değişken (özellikler/features) arasındaki ilişkiyi modellemek ve tahminlemek için kullanılan denetimli bir makine öğrenmesi yöntemidir.

### Temel Türleri:
1. **Basit Doğrusall Regresyon (Simple Linear Regression):** Tek bir bağımsız değişken kullanarak hedefi tahmin eder. Formülü:
   $$y = \beta_0 + \beta_1x + \epsilon$$
   - $y$: Bağımlı değişken (Ev Fiyatı)
   - $x$: Bağımsız değişken (Metrekare)
   - $\beta_0$: Kesişim noktası (Intercept)
   - $\beta_1$: Eğim katsayısı (Slope)
   
2. **Çoklu Doğrusal Regresyon (Multiple Linear Regression):** Birden fazla bağımsız değişken kullanarak hedefi tahmin eder. Formülü:
   $$y = \beta_0 + \beta_1x_1 + \beta_2x_2 + ... + \beta_nx_n + \epsilon$$
   - $x_1, x_2, ..., x_n$: Evin metrekaresi, oda sayısı, yaşı gibi farklı özellikler.

### Model Değerlendirme Metrikleri:
- **R² (Belirlenlik Katsayısı):** Bağımsız değişkenlerin, hedef değişkendeki değişimin yüzde kaçını açıkladığını gösterir. 1'e yakın olması modelin başarısını gösterir.
- **RMSE (Root Mean Squared Error - Hata Kareler Ortalamasının Karekökü):** Tahminlerin gerçek değerlerden ne kadar saptığının ortalama ölçüsüdür. Büyük hataları daha sert cezalandırır.
- **MAE (Mean Absolute Error - Ortalama Mutlak Hata):** Tahmin hatalarının mutlak değerlerinin ortalamasıdır.

---
## 2. Gerekli Kütüphanelerin Yüklenmesi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
print("Kütüphaneler başarıyla yüklendi.")

---
## 3. Veri Setinin Yüklenmesi ve Keşfi

Bu çalışmada, makine öğrenmesinde ev fiyatlarını tahmin etmek için sıkça kullanılan **California Housing** veri setinin sadeleştirilmiş bir versiyonunu kullanacağız. 
- **Hedef Değişken (Target):** `MedHouseValue` (Evlerin Ortalama Değeri - 100.000$ bazında)
- **Özellikler (Features):** `MedInc` (Bölgedeki Ortalama Gelir), `HouseAge` (Ev Yaşı), `AveRooms` (Ortalama Oda Sayısı)

In [ ]:
# California Housing veri setini doğrudan güvenilir bir kaynaktan çekiyoruz
url = "https://raw.githubusercontent.com/sonwand/Celebrity-Housing-Prices-Prediction/master/California_Houses.csv"
full_df = pd.read_csv(url)

# Analizimiz için gerekli sütunları seçip Türkçeleştirelim veya hocanın formatına uygun sadeleştirelim
# Orijinal sütunlardan temsili bir sadeleştirme yapıyoruz:
df = full_df[['Median_Income', 'Median_Age', 'Tot_Rooms', 'Median_House_Value']].copy()
df.columns = ['Ortalama_Gelir', 'Ev_Yasi', 'Toplam_Oda', 'Ev_Degeri']

# Veriyi çok büyük olmaması adına ilk 1000 satırla sınırlayalım (Analizin netliği için)
df = df.head(1000)

print("Veri Seti İlk 5 Satır:")
display(df.head())

print(f"\nVeri Seti Boyutu: {df.shape}")

### Korelasyon Analizi
Değişkenlerin birbirleriyle olan ilişkilerini inceleyelim.

In [ ]:
correlation_matrix = df.corr()
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='RdBu_r', fmt='.2f', vmin=-1, vmax=1)
plt.title('Değişkenler Arası Korelasyon Matrisi')
plt.show()

---
## 4. Basit Doğrusal Regresyon (Simple Linear Regression)

Ev değerini tahmin etmek için ev değeriyle en yüksek korelasyona sahip olan `Ortalama_Gelir` değişkenini seçiyoruz.

In [ ]:
X_simple = df[['Ortalama_Gelir']]
y = df['Ev_Degeri']

# %80 Eğitim, %20 Test olarak ayırma
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_simple, y, test_size=0.2, random_state=42)

# Modelin kurulması ve eğitilmesi
simple_model = LinearRegression()
simple_model.fit(X_train_s, y_train_s)

beta_0 = simple_model.intercept_
beta_1 = simple_model.coef_[0]

print(f"Model Denklemimiz: Ev_Degeri = {beta_0:.2f} + {beta_1:.2f} × Ortalama_Gelir")

### Basit Regresyon Model Performansı

In [ ]:
y_pred_test_s = simple_model.predict(X_test_s)

r2_test_s = r2_score(y_test_s, y_pred_test_s)
rmse_test_s = np.sqrt(mean_squared_error(y_test_s, y_pred_test_s))
mae_test_s = mean_absolute_error(y_test_s, y_pred_test_s)

print(f"Basit Regresyon Test R² Skoru: {r2_test_s:.3f}")
print(f"Basit Regresyon Test RMSE     : {rmse_test_s:.2f}")
print(f"Basit Regresyon Test MAE      : {mae_test_s:.2f}")

### Regresyon Doğrusunun Çizilmesi

In [ ]:
plt.scatter(X_test_s, y_test_s, color='gray', alpha=0.5, label='Gerçek Değerler')
plt.plot(X_test_s, y_pred_test_s, color='red', linewidth=2, label='Regresyon Doğrusu')
plt.xlabel('Bölge Ortalama Geliri')
plt.ylabel('Ev Değeri ($)')
plt.title('Basit Doğrusal Regresyon - Gelire Göre Ev Fiyatı Tahmini')
plt.legend()
plt.show()

---
## 5. Çoklu Doğrusal Regresyon (Multiple Linear Regression)

Şimdi tahmini güçlendirmek için `Ev_Yasi` ve `Toplam_Oda` bilgilerini de modele ekleyerek çoklu regresyon yapıyoruz.

In [ ]:
X_multi = df[['Ortalama_Gelir', 'Ev_Yasi', 'Toplam_Oda']]

# Aynı random_state ile veriyi bölme
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_multi, y, test_size=0.2, random_state=42)

# Çoklu modelin eğitilmesi
multi_model = LinearRegression()
multi_model.fit(X_train_m, y_train_m)

print("Çoklu Doğrusal Regresyon katsayıları:")
for col, coef in zip(X_multi.columns, multi_model.coef_):
    print(f"  {col}: {coef:.2f}")
print(f"  Intercept (Kesişim): {multi_model.intercept_:.2f}")

### Çoklu Regresyon Model Performansı

In [ ]:
y_pred_test_m = multi_model.predict(X_test_m)

r2_test_m = r2_score(y_test_m, y_pred_test_m)
rmse_test_m = np.sqrt(mean_squared_error(y_test_m, y_pred_test_m))
mae_test_m = mean_absolute_error(y_test_m, y_pred_test_m)

print(f"Çoklu Regresyon Test R² Skoru: {r2_test_m:.3f}")
print(f"Çoklu Regresyon Test RMSE     : {rmse_test_m:.2f}")
print(f"Çoklu Regresyon Test MAE      : {mae_test_m:.2f}")

---
## 6. Model Karşılaştırması ve Sonuçlar

Hocamızın notebook'undaki raporlama formatına uygun olarak her iki modeli yan yana karşılaştıralım.

In [ ]:
print('='*60)
print('REGRESYON MODELLERİ KARŞILAŞTIRMA RAPORU')
print('='*60)
print(f'1. BASİT DOĞRUSAL REGRESYON')
print(f'   Hedef: Ev_Degeri')
print(f'   Özellik: Ortalama_Gelir')
print(f'   Denklem: Ev_Degeri = {beta_0:.2f} + {beta_1:.2f} × Ortalama_Gelir')
print(f'   Test R²: {r2_test_s:.3f}')
print(f'   Test RMSE: {rmse_test_s:.2f}')
print(f'   Test MAE: {mae_test_s:.2f}')
print(f'\n')
print(f'2. ÇOKLU DOĞRUSAL REGRESYON')
print(f'   Hedef: Ev_Degeri')
print(f'   Özellikler: [\'Ortalama_Gelir\', \'Ev_Yasi\', \'Toplam_Oda\']')
print(f'   Test R²: {r2_test_m:.3f}')
print(f'   Test RMSE: {rmse_test_m:.2f}')
print(f'   Test MAE: {mae_test_m:.2f}')
print(f'\n')
print(f'3. MODEL KARŞILAŞTIRMASI')
better_model = 'Çoklu Regresyon' if r2_test_m > r2_test_s else 'Basit Regresyon'
print(f'   Daha Yüksek R² Değerine Sahip Model: {better_model}')
print('='*60)